En este codigo se entrena de nuevo el modelo de microsoft, pero antes se le da un mayor peso (a partir de una distribucion ponderada) a las clases de menor aparición, para que, después de este preprocesado, todas las clases tenga la misma relevancia, sin importar el numero de apariciones.

Para simplificar hemos decidido también mapear "calmado" a "neutral" ya que era complicado diferencias ambas clases.

In [ ]:
!pip install -q transformers datasets evaluate accelerate librosa soundfile scikit-learn torchcodec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 100.3 MB/s eta 0:00:00


In [ ]:
import os
import zipfile
import glob
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, Audio
from transformers import AutoModelForAudioClassification, Wav2Vec2FeatureExtractor, TrainingArguments, Trainer
import evaluate
from sklearn.model_selection import train_test_split
from google.colab import drive
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [ ]:
# 2. Mount Drive to save the final model
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/RAVDESS_WavLM_Final"

# --- CONFIGURATION ---
MODEL_ID = "microsoft/wavlm-base-plus"

# RAVDESS Filename Identifiers
# 01 = neutral, 02 = calm, 03 = happy, 04 = sad,
# 05 = angry, 06 = fearful, 07 = disgust, 08 = surprised
emotion_map = {
    '01': 'neutral',
    '02': 'neutral', # calm -> neutral to simplify
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fear',
    '07': 'disgust',
    '08': 'surprise'
}

# Define the labels for the model config
labels = sorted(list(set(emotion_map.values())))
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

print(f"Labels: {labels}")

# --- 3. DOWNLOAD & UNZIP DATASET ---
if not os.path.exists("Audio_Speech_Actors_01-24.zip"):
    print("Downloading RAVDESS dataset...")
    # Using wget to download from Zenodo
    !wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip

if not os.path.exists("./ravdess_data"):
    print("Unzipping dataset...")
    with zipfile.ZipFile("Audio_Speech_Actors_01-24.zip", 'r') as zip_ref:
        zip_ref.extractall("./ravdess_data")

Mounted at /content/drive
Labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Unzipping dataset...


In [ ]:
print("Parsing files...")
file_paths = []
file_labels = []

# Scan all folders for .wav files
for file in glob.glob("./ravdess_data/Actor_*/*.wav"):
    filename = os.path.basename(file)
    parts = filename.split("-")

    if len(parts) < 3: continue

    emotion_code = parts[2] 

    emotion_label = emotion_map.get(emotion_code)
    if emotion_label:
        file_paths.append(file)
        file_labels.append(label2id[emotion_label])

# Create a Pandas DataFrame
df = pd.DataFrame({"file": file_paths, "label": file_labels})

# Split into Train (80%) and Test (20%)
# We use 'stratify' to ensure all emotions are represented equally in both sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(test_df)

Parsing files...


In [ ]:
# --- 5. AUDIO PROCESSING ---
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

# Converts the 'file' column into actual audio arrays
train_dataset = train_dataset.cast_column("file", Audio(sampling_rate=16000))
eval_dataset = eval_dataset.cast_column("file", Audio(sampling_rate=16000))

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["file"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        max_length=16000 * 5, 
        truncation=True
    )
    return inputs

print("Preprocessing dataset (This might take a minute)...")
encoded_train = train_dataset.map(preprocess_function, batched=True, batch_size=4, remove_columns=["file", "__index_level_0__"])
encoded_eval = eval_dataset.map(preprocess_function, batched=True, batch_size=4, remove_columns=["file", "__index_level_0__"])

@dataclass
class DataCollatorForAudioClassification:
    """
    Data collator that will dynamically pad the inputs received.
    """
    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        labels = [feature["label"] for feature in features]

        # Use the feature extractor to pad the audio inputs (batching)
        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )

        # Convert labels to a tensor of long integers
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Preprocessing dataset (This might take a minute)...


Map:   0%|          | 0/1152 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

In [ ]:
# --- 6. MODEL SETUP ---
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)
model.freeze_feature_encoder()

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    f1 = f1_metric.compute(predictions=predictions, references=eval_pred.label_ids, average="weighted")
    return {**acc, **f1}

# --- 7. TRAINING ---
training_args = TrainingArguments(
    output_dir="./wavlm_ravdess",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    save_total_limit=2,
    report_to="none"
)

# Initialize Trainer with the Data Collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorForAudioClassification(feature_extractor=feature_extractor)
)

print("Starting Training...")
trainer.train()

# --- 8. SAVE MODEL ---
print(f"Saving final model to {SAVE_PATH}...")
trainer.save_model(SAVE_PATH)
feature_extractor.save_pretrained(SAVE_PATH)
print("Done!")

Some weights of WavLMForSequenceClassification were not initialized from the model checkpoint at microsoft/wavlm-base-plus and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2076850093.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting Training...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.943800,1.822489,0.326389,0.206545
2,1.815900,1.760145,0.315972,0.228687
3,1.485500,1.336407,0.569444,0.539773
4,1.343600,1.171794,0.680556,0.676666
5,1.065500,1.032789,0.708333,0.681545
6,0.975200,0.879311,0.791667,0.783196
7,0.815100,0.846566,0.798611,0.802594
8,0.736800,0.764032,0.829861,0.830797
9,0.666800,0.720913,0.861111,0.861595
10,0.620600,0.696600,0.871528,0.873442


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Saving final model to /content/drive/MyDrive/RAVDESS_WavLM_Final...
Done!


In [ ]:
import random
import librosa
MODEL_PATH = "/content/drive/MyDrive/RAVDESS_WavLM_Final"
print(f"Loading model from {MODEL_PATH}...")

model = AutoModelForAudioClassification.from_pretrained(MODEL_PATH)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)

# 2. Define the prediction function
def predict_emotion(audio_path):
    # Load audio and force it to 16000 Hz (WavLM requirement)
    audio, sr = librosa.load(audio_path, sr=16000)

    # Process the audio
    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        padding=True,
        return_tensors="pt"
    )

    # Get the prediction
    with torch.no_grad():
        logits = model(**inputs).logits

    # Find the label with the highest score
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = model.config.id2label[predicted_id]

    return predicted_label

# 3. Pick a random file from RAVDESS to test
all_files = glob.glob("./ravdess_data/Actor_*/*.wav")
random_file = random.choice(all_files)

# Get the ACTUAL label from the filename (Filename format: 03-01-XX-...)
# 03 = Happy, 04 = Sad, 05 = Angry, etc.
emotion_map = {
    '01': 'neutral', '02': 'neutral', '03': 'happy',
    '04': 'sad', '05': 'angry', '06': 'fear',
    '07': 'disgust', '08': 'surprise'
}
filename = os.path.basename(random_file)
true_code = filename.split("-")[2]
true_label = emotion_map.get(true_code)

# 4. Run Prediction
print(f"\n--- Testing on file: {filename} ---")
predicted_label = predict_emotion(random_file)

print(f"True Emotion:      {true_label.upper()}")
print(f"Model Prediction:  {predicted_label.upper()}")

if true_label == predicted_label:
    print("CORRECT!")
else:
    print("INCORRECT")

Loading model from /content/drive/MyDrive/RAVDESS_WavLM_Final...

--- Testing on file: 03-01-04-01-01-01-12.wav ---


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


True Emotion:      SAD
Model Prediction:  SAD
✅ CORRECT!


Los resultados obtenidos son significativamente superiores, con un training loss de 0.620600, un val loss de 0.696600, una acc de 0.871528. y una f1 0.873442. El ponderar los pesos para aumentar la importancia de las clases que aparecen menos ha dado muy buenos resultados comparado con el primer intento de entrenar el modelo.